# Step 3: Keyword Extraction & Named Entity Recognition (NER)
## YouTube Fast Fashion Intelligence Engine — CSCI370

### What are we doing in this step?

We are extracting two types of information from each comment:

---

**1. Keywords (via KeyBERT)**

KeyBERT finds the most important phrases in a comment.
For example, from *"Shein workers are paid almost nothing and work in dangerous conditions"*,
KeyBERT might extract: `["shein workers", "dangerous conditions", "paid nothing"]`

These keywords will power the **word clouds** in your dashboard and help surface trending topics.

**How it works under the hood:**
KeyBERT converts your comment into a vector (list of numbers representing meaning),
then finds which phrases are most *similar* to the overall meaning of the comment.
It uses a model called `all-MiniLM-L6-v2` from sentence-transformers.

---

**2. Named Entity Recognition / NER (via spaCy)**

spaCy reads each comment and labels specific words by type:
- `ORG` → Organizations / brands (SHEIN, Temu, Zara, H&M)
- `GPE` → Countries / cities (Bangladesh, China, USA)
- `PERSON` → People's names
- `MONEY` → Dollar amounts, prices

For example, from *"Shein ships from China and pays workers $1/day"*,
spaCy extracts: `SHEIN (ORG)`, `China (GPE)`, `$1/day (MONEY)`

These entities will power the **brand mention charts** and **geographic heatmaps** in your dashboard.


## 0. Install Required Libraries

Run this cell once. It may take 1-2 minutes to download the spaCy model.

In [1]:
!pip install spacy keybert sentence-transformers -q
!python -m spacy download en_core_web_sm


[notice] A new release of pip is available: 25.2 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


Defaulting to user installation because normal site-packages is not writeable
     ---------------------------------------- 0.0/12.8 MB ? eta -:--:--
     ---- ----------------------------------- 1.3/12.8 MB 7.4 MB/s eta 0:00:02
     ------- -------------------------------- 2.4/12.8 MB 6.9 MB/s eta 0:00:02
     --------- ------------------------------ 2.9/12.8 MB 4.6 MB/s eta 0:00:03
     --------- ------------------------------ 2.9/12.8 MB 4.6 MB/s eta 0:00:03
     --------- ------------------------------ 3.1/12.8 MB 3.0 MB/s eta 0:00:04
     ------------ --------------------------- 3.9/12.8 MB 3.1 MB/s eta 0:00:03
     -------------- ------------------------- 4.7/12.8 MB 3.1 MB/s eta 0:00:03
     ------------------ --------------------- 6.0/12.8 MB 3.5 MB/s eta 0:00:02
     --------------------- ------------------ 6.8/12.8 MB 3.5 MB/s eta 0:00:02
     ---------------------- ----------------- 7.1/12.8 MB 3.4 MB/s eta 0:00:02
     ------------------------- -------------- 8.1/12.8 MB 3.


[notice] A new release of pip is available: 25.2 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


## 1. Load Libraries and Data

In [2]:
import pandas as pd
import spacy
from keybert import KeyBERT
import ast

# Load the English-only dataset from Step 2b
df = pd.read_csv('youtube_comments_english.csv')

print(f"Loaded {len(df):,} comments")
print(f"Columns: {df.columns.tolist()}")
df.head(3)

C:\Users\AK Khan\AppData\Roaming\Python\Python313\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loaded 30,609 comments
Columns: ['author', 'updated_at', 'like_count', 'text', 'video_id', 'public', 'text_clean', 'sentiment_score', 'sentiment_label', 'sentiment_source', 'language']


,author,updated_at,like_count,text,video_id,public,text_clean,sentiment_score,sentiment_label,sentiment_source,language
0,@joaquinrodriguezvillegas366,2026-02-25 00:49:20+00:00,0,This doesn't surprise me since the entire hist...,qpClEvsjW0s,True,This doesn't surprise me since the entire hist...,-0.0121,Negative,RoBERTa,en
1,@andrerosekriel1127,2026-02-22 11:15:55+00:00,0,Okay then why work for these factories.....or ...,qpClEvsjW0s,True,Okay then why work for these factories.....or ...,-0.2732,Negative,VADER,en
2,@sbradshaw1886,2026-03-17 02:12:15+00:00,0,Please do not be this ignorant. The US and oth...,qpClEvsjW0s,True,Please do not be this ignorant. The US and oth...,-0.0194,Negative,RoBERTa,en


## 2. Load the NLP Models

We load two models:
- **spaCy `en_core_web_sm`** — a small but fast English NLP model trained on news/web text
- **KeyBERT with `all-MiniLM-L6-v2`** — a lightweight sentence embedding model

Both are loaded once and reused for every comment.


In [3]:
# Load spaCy model — used for NER
nlp = spacy.load('en_core_web_sm')

# Load KeyBERT model — used for keyword extraction
# all-MiniLM-L6-v2 is fast and accurate for short social media text
kw_model = KeyBERT(model='all-MiniLM-L6-v2')

print("Both models loaded successfully!")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1157.18it/s]


Both models loaded successfully!


## 2b. Fix: Add Custom Brand Entity Rules

**The Problem:**

spaCy's `en_core_web_sm` model was trained on news articles from years ago.
It has never seen brands like *Shein*, *Temu*, or *Zara* — so it guesses them
as `PERSON` (because they look like names) instead of `ORG`.

Example without fix:
```
"I bought from Shein" → Shein (PERSON) ❌
"Temu is so cheap"   → Temu  (PERSON) ❌
```

**The Fix: `EntityRuler`**

spaCy has a built-in component called `EntityRuler` that lets you define custom rules.
We give it a list of brand names and tell it: *"whenever you see this word, always label it ORG"*.

The `EntityRuler` runs **before** the regular NER model, so our rules take priority.
This is called **rule-based NLP** — combining hand-crafted rules with a machine learning model.


In [4]:
# ── Custom Brand Rules ────────────────────────────────────────────────────────
# These are all the fast fashion / related brands we expect in our dataset.
# Add more here if you find others in the data.

FASHION_BRANDS = [
    # Ultra fast fashion
    "shein", "temu", "romwe", "zaful", "fashionova", "fashion nova",
    # High street fast fashion  
    "zara", "h&m", "hm", "h & m", "primark", "topshop", "forever 21",
    "forever21", "urban outfitters", "asos", "boohoo", "prettylittlething",
    "plt", "missguided", "nasty gal",
    # Department / general
    "uniqlo", "gap", "old navy", "target", "walmart",
    # Luxury (sometimes mentioned for comparison)
    "gucci", "louis vuitton", "prada", "dior",
]

# Add the EntityRuler to the spaCy pipeline
# patterns tell spaCy: "if you see this text, label it as ORG"
ruler = nlp.add_pipe("entity_ruler", before="ner")

patterns = []
for brand in FASHION_BRANDS:
    patterns.append({
        "label": "ORG",
        "pattern": brand          # matches lowercase
    })
    patterns.append({
        "label": "ORG",
        "pattern": brand.title()  # matches Title Case e.g. "Shein"
    })
    patterns.append({
        "label": "ORG",
        "pattern": brand.upper()  # matches ALL CAPS e.g. "SHEIN"
    })

ruler.add_patterns(patterns)

print(f"Added {len(patterns)} brand patterns to spaCy pipeline")
print(f"Pipeline order: {nlp.pipe_names}")

# ── Verify the fix works ──────────────────────────────────────────────────────
test_comments = [
    "I bought from Shein last week and the quality was terrible",
    "TEMU is destroying small businesses with their cheap prices",
    "Both Zara and H&M are fast fashion brands but at least they have sustainability reports",
    "shein workers are paid almost nothing according to this documentary"
]

print("\nVerification — entity labels after fix:")
for comment in test_comments:
    doc = nlp(comment)
    entities = [(ent.text, ent.label_) for ent in doc.ents]
    print(f"\n  Text     : {comment}")
    print(f"  Entities : {entities}")

Added 90 brand patterns to spaCy pipeline
Pipeline order: ['tok2vec', 'tagger', 'parser', 'attribute_ruler', 'lemmatizer', 'entity_ruler', 'ner']

Verification — entity labels after fix:

  Text     : I bought from Shein last week and the quality was terrible
  Entities : [('Shein', 'ORG'), ('last week', 'DATE')]

  Text     : TEMU is destroying small businesses with their cheap prices
  Entities : [('TEMU', 'ORG')]

  Text     : Both Zara and H&M are fast fashion brands but at least they have sustainability reports
  Entities : [('Zara', 'ORG'), ('H&M', 'ORG')]

  Text     : shein workers are paid almost nothing according to this documentary
  Entities : [('shein', 'ORG')]


## 3. See How Both Tools Work — Live Demo

Before running on 30,000 comments, let's see exactly what each tool produces
on a few example sentences so we fully understand the output.


In [5]:
demo_comments = [
    "Shein ships everything from China and pays garment workers almost nothing, it's disgusting",
    "I love Temu's prices but I feel guilty about the environment and the packaging waste",
    "H&M and Zara are just as bad as Shein when it comes to fast fashion and labor conditions in Bangladesh"
]

print("=" * 70)
print("KEYBERT — Keyword Extraction")
print("=" * 70)
for comment in demo_comments:
    keywords = kw_model.extract_keywords(
        comment,
        keyphrase_ngram_range=(1, 2),  # extract 1 or 2 word phrases
        stop_words='english',          # ignore words like 'the', 'is', 'and'
        top_n=3                        # get top 3 keywords per comment
    )
    print(f"\nComment : {comment[:80]}...")
    print(f"Keywords: {[kw for kw, score in keywords]}")

print("\n" + "=" * 70)
print("SPACY — Named Entity Recognition")
print("=" * 70)
for comment in demo_comments:
    doc = nlp(comment)
    entities = [(ent.text, ent.label_) for ent in doc.ents]
    print(f"\nComment  : {comment[:80]}...")
    print(f"Entities : {entities}")

KEYBERT — Keyword Extraction

Comment : Shein ships everything from China and pays garment workers almost nothing, it's ...
Keywords: ['china pays', 'pays garment', 'ships china']

Comment : I love Temu's prices but I feel guilty about the environment and the packaging w...
Keywords: ['temu prices', 'love temu', 'temu']

Comment : H&M and Zara are just as bad as Shein when it comes to fast fashion and labor co...
Keywords: ['conditions bangladesh', 'bangladesh', 'fashion labor']

SPACY — Named Entity Recognition

Comment  : Shein ships everything from China and pays garment workers almost nothing, it's ...
Entities : [('Shein', 'ORG'), ('China', 'GPE')]

Comment  : I love Temu's prices but I feel guilty about the environment and the packaging w...
Entities : [('Temu', 'ORG')]

Comment  : H&M and Zara are just as bad as Shein when it comes to fast fashion and labor co...
Entities : [('H&M', 'ORG'), ('Zara', 'ORG'), ('Shein', 'ORG'), ('Bangladesh', 'GPE')]


## 4. Run NER on All 30,609 Comments

NER with spaCy is fast — it processes about 10,000 comments per minute.
We extract all entities and store them as a list per comment.

We also create a focused column that extracts ONLY the entity types
most relevant to our fast fashion project: brands (ORG) and locations (GPE).


In [6]:
# Function to extract named entities from one comment
def extract_entities(text):
    """Returns a list of (entity_text, entity_type) tuples."""
    doc = nlp(str(text))
    return [(ent.text, ent.label_) for ent in doc.ents]

# Function to extract only ORG and GPE entities (most relevant for our project)
def extract_brands_locations(text):
    """Returns dict with 'brands' (ORG) and 'locations' (GPE) lists."""
    doc = nlp(str(text))
    brands    = [ent.text for ent in doc.ents if ent.label_ == 'ORG']
    locations = [ent.text for ent in doc.ents if ent.label_ == 'GPE']
    return {'brands': brands, 'locations': locations}

print("Running NER on all comments... (takes ~3-4 minutes)")
df['entities']    = df['text_clean'].apply(extract_entities)
df['brands']      = df['text_clean'].apply(lambda t: extract_brands_locations(t)['brands'])
df['locations']   = df['text_clean'].apply(lambda t: extract_brands_locations(t)['locations'])
print("NER complete!")

# Quick preview
print("\nSample NER output:")
sample = df[df['brands'].apply(len) > 0][['text_clean', 'brands', 'locations']].head(5)
for _, row in sample.iterrows():
    print(f"\n  Text      : {row['text_clean'][:100]}")
    print(f"  Brands    : {row['brands']}")
    print(f"  Locations : {row['locations']}")

Running NER on all comments... (takes ~3-4 minutes)
NER complete!

Sample NER output:

  Text      : This doesn't surprise me since the entire history of capitalism from the 19th century to today is th
  Brands    : ['Das Kapital', 'HORROR']
  Locations : []

  Text      : They fuel buying addiction. Same with Temu. Not to mention the waste they are creating.
  Brands    : ['Temu']
  Locations : []

  Text      : Shein are SCAMMERS and fraudsters
  Brands    : ['Shein']
  Locations : []

  Text      : What about H&M ?
  Brands    : ['H&M']
  Locations : []

  Text      : Perhaps women wearing Shein products should pay reparations to the grossly mistreated workers.
  Brands    : ['Shein']
  Locations : []


## 5. What Are the Most Mentioned Brands and Locations?

Let's count the most frequently mentioned brands and locations across all comments.
This is a key insight for your dashboard.


In [7]:
from collections import Counter

# Flatten all brand mentions into one big list and count
all_brands = [brand.lower() for brands_list in df['brands'] for brand in brands_list]
brand_counts = Counter(all_brands).most_common(20)

print("Top 20 Most Mentioned Brands/Organizations:")
for brand, count in brand_counts:
    print(f"  {brand:25s}: {count:,}")

print()

# Flatten all location mentions and count
all_locations = [loc.lower() for locs_list in df['locations'] for loc in locs_list]
location_counts = Counter(all_locations).most_common(20)

print("Top 20 Most Mentioned Locations:")
for loc, count in location_counts:
    print(f"  {loc:25s}: {count:,}")

Top 20 Most Mentioned Brands/Organizations:
  shein                    : 3,874
  zara                     : 680
  h&m                      : 604
  temu                     : 422
  gap                      : 174
  walmart                  : 159
  amazon                   : 141
  target                   : 112
  uniqlo                   : 112
  primark                  : 110
  hot topic                : 90
  shien                    : 86
  🙏                        : 78
  nike                     : 75
  forever 21               : 74
  ccp                      : 71
  gucci                    : 67
  hm                       : 60
  youtube                  : 55
  tiktok                   : 54

Top 20 Most Mentioned Locations:
  china                    : 1,455
  bangladesh               : 667
  us                       : 533
  india                    : 391
  america                  : 312
  usa                      : 183
  alaska                   : 155
  italy                    : 141
  uk

## 6. Run Keyword Extraction on All 30,609 Comments

KeyBERT is slower than spaCy — expect about 15-25 minutes for 30k comments
because it runs a neural embedding model on each comment.

We extract the top 3 keyword phrases per comment.


In [8]:
# Function to extract keywords from one comment
def extract_keywords(text):
    """Returns list of top 3 keyword phrases."""
    try:
        keywords = kw_model.extract_keywords(
            str(text),
            keyphrase_ngram_range=(1, 2),  # 1 or 2 word phrases
            stop_words='english',
            top_n=3
        )
        return [kw for kw, score in keywords]
    except Exception:
        return []

print("Running keyword extraction on all comments...")
print("This may take 15-25 minutes — go grab a coffee!")
df['keywords'] = df['text_clean'].apply(extract_keywords)
print("Keyword extraction complete!")

# Preview
print("\nSample keywords:")
sample = df[df['keywords'].apply(len) > 0][['text_clean','keywords']].head(5)
for _, row in sample.iterrows():
    print(f"\n  Text    : {row['text_clean'][:100]}")
    print(f"  Keywords: {row['keywords']}")

Running keyword extraction on all comments...
This may take 15-25 minutes — go grab a coffee!
Keyword extraction complete!

Sample keywords:

  Text    : This doesn't surprise me since the entire history of capitalism from the 19th century to today is th
  Keywords: ['workers imperialism', 'capitalism 19th', 'understand capitalism']

  Text    : Okay then why work for these factories.....or are you forced to be there to work?
  Keywords: ['work factories', 'factories forced', 'factories']

  Text    : Please do not be this ignorant. The US and other countries are called "first-world" for a reason.
  Keywords: ['countries called', 'ignorant countries', 'called world']

  Text    : @sbradshaw1886 Still a choice ...Ignorence, really!
  Keywords: ['sbradshaw1886', 'sbradshaw1886 choice', 'ignorence']

  Text    : They fuel buying addiction. Same with Temu. Not to mention the waste they are creating.
  Keywords: ['addiction temu', 'temu', 'temu mention']


## 7. What Are the Most Common Keywords Overall?

Let's look at the top keywords across the entire dataset.
These will directly feed your word cloud visualization in the dashboard.


In [9]:
# Flatten and count all keywords
all_keywords = [kw for kw_list in df['keywords'] for kw in kw_list]
keyword_counts = Counter(all_keywords).most_common(25)

print("Top 25 Keywords Across All Comments:")
for kw, count in keyword_counts:
    print(f"  {kw:30s}: {count:,}")

Top 25 Keywords Across All Comments:
  fast fashion                  : 1,012
  fashion                       : 692
  thrift                        : 317
  clothes                       : 305
  buy clothes                   : 284
  shein                         : 264
  bangladesh                    : 256
  thrift stores                 : 236
  hot topic                     : 230
  hunter                        : 209
  buying clothes                : 160
  temu                          : 156
  video                         : 141
  new clothes                   : 136
  thank                         : 133
  papa meat                     : 128
  fashion brands                : 124
  fashion industry              : 121
  clothing                      : 120
  hollister                     : 120
  buy shein                     : 114
  shop                          : 111
  zara                          : 110
  slavery                       : 109
  jeans                         : 103


## 8. Keywords Broken Down by Sentiment

This is a powerful insight — what words appear most in NEGATIVE comments vs POSITIVE?


In [10]:
for label in ['Positive', 'Negative', 'Neutral']:
    subset = df[df['sentiment_label'] == label]
    kws = [kw for kw_list in subset['keywords'] for kw in kw_list]
    top = Counter(kws).most_common(10)
    print(f"\nTop 10 keywords in {label.upper()} comments:")
    for kw, count in top:
        print(f"  {kw:30s}: {count:,}")


Top 10 keywords in POSITIVE comments:
  fast fashion                  : 459
  fashion                       : 309
  thrift                        : 176
  buy clothes                   : 163
  clothes                       : 145
  thank                         : 132
  hunter                        : 126
  thrift stores                 : 114
  video                         : 110
  bangladesh                    : 92

Top 10 keywords in NEGATIVE comments:
  fast fashion                  : 446
  fashion                       : 323
  bangladesh                    : 133
  hot topic                     : 122
  shein                         : 107
  clothes                       : 98
  slavery                       : 93
  buy clothes                   : 86
  thrift stores                 : 84
  thrift                        : 82

Top 10 keywords in NEUTRAL comments:
  fast fashion                  : 107
  shein                         : 65
  clothes                       : 62
  fashion         

## 9. Save the Results

In [11]:
# Save the dataset with all new columns added
output_path = 'youtube_comments_ner_keywords.csv'
df.to_csv(output_path, index=False)

print(f"Saved to: {output_path}")
print(f"Total rows: {len(df):,}")
print(f"Columns: {df.columns.tolist()}")
print("\nPreview of new columns:")
df[['text_clean', 'brands', 'locations', 'keywords']].head(3)

Saved to: youtube_comments_ner_keywords.csv
Total rows: 30,609
Columns: ['author', 'updated_at', 'like_count', 'text', 'video_id', 'public', 'text_clean', 'sentiment_score', 'sentiment_label', 'sentiment_source', 'language', 'entities', 'brands', 'locations', 'keywords']

Preview of new columns:


,text_clean,brands,locations,keywords
0,This doesn't surprise me since the entire hist...,"[Das Kapital, HORROR]",[],"[workers imperialism, capitalism 19th, underst..."
1,Okay then why work for these factories.....or ...,[],[],"[work factories, factories forced, factories]"
2,Please do not be this ignorant. The US and oth...,[],[US],"[countries called, ignorant countries, called ..."


## Summary

**What we did in this step:**
- Used **spaCy** (`en_core_web_sm`) to run Named Entity Recognition on all 30,609 comments
- Extracted `brands` (ORG entities) and `locations` (GPE entities) per comment
- Used **KeyBERT** (`all-MiniLM-L6-v2`) to extract top 3 keyword phrases per comment
- Analyzed most mentioned brands, locations, and keywords overall and by sentiment

**New columns added:**
- `entities` — full list of all (text, type) entity pairs
- `brands` — list of brand/org mentions per comment
- `locations` — list of country/city mentions per comment
- `keywords` — top 3 keyword phrases per comment

**These columns will power in the dashboard:**
- Word clouds (from `keywords`)
- Brand mention bar charts (from `brands`)
- Geographic mention charts (from `locations`)
- Per-sentiment keyword comp
